# Imports

In [ ]:
import numpy as np
from datetime import datetime
import datetime
from google.colab import drive
drive.mount('/content/drive')
from statsmodels.tsa import holtwinters as hw
import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.express as px
import plotly
import plotly.graph_objs as go

FREQ = 'QS'

Mounted at /content/drive


In [ ]:
#Установка необходимых пакетов для доступа к SPDF, в коллабе запускать каждый раз
!pip install pwlf
!pip install -U xarray
!pip install -U cdflib
!pip install -U cdasws

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for pwlf: filename=pwlf-2.2.1-py3-none-any.whl size=16605 sha256=3719c6aa3208cf07053ee318d1a8318d72a1b359094773bc3136506bf546b593
  Stored in directory: /root/.cache/pip/wheels/d9/13/6f/a9201ce279d71065ce782d82418d7c6877be6bb818ae0d1095
  Created wheel for pyDOE: filename=pyDOE-0.3.8-py3-none-any.whl size=18168 sha256=318b7cd5625d3dccd72719dedb155c48ec3947cf34a57d7fcd040cf6cd37bff1
  Stored in directory: /root/.cache/pip/wheels/ce/b6/d7/c6b64746dba6433c593e471e0ac3acf4f36040456d1d160d17
Successfully built pwlf pyDOE
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.0 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2023.7.0
    Uninstalling xarray-2023.7.0:
      Successfully uninstalled xarray-2023.7.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.2/229.2 kB 4.9 MB/s eta 0:00:00


In [ ]:
from cdasws import CdasWs
from datetime import timedelta
from datetime import timezone
from datetime import datetime
from IPython.display import clear_output

# Variables

In [ ]:
#Промежуток для которого ведутся расчёты в формате "yyyy-mm-dd hh:MM:SSZ"
#Z - Обозначает UTC таймзону
start_date = "2000-01-01 00:00:00Z"
end_date = "2000-01-02 01:05:00Z"

#путь
path = '/content/drive/My Drive/export.csv'

# Main Code

In [ ]:
date_start = datetime.strptime(start_date, "%Y-%m-%d %H:%M:%S%z")
date_end = datetime.strptime(end_date, "%Y-%m-%d %H:%M:%S%z")

In [ ]:
def current_sheet_detection(date_start, date_end, path):
  cdas = CdasWs()
  vars = ['BF1']
  df = pd.DataFrame()
  days_num = (date_end - date_start).days
  print('started MFI calc this may take some time (30-60 sec/day)')
  for i in range(days_num + 1):
    if i != days_num:
      status, data = cdas.get_data('WI_H2_MFI', vars, date_start + pd.DateOffset(days=i), date_start + pd.DateOffset(days=i+1))
    else:
      status, data = cdas.get_data('WI_H2_MFI', vars, date_start + pd.DateOffset(days=i), date_end)
    if data is not None:
      df_temp = data.to_dataframe()
      df_temp['date'] = pd.to_datetime(df_temp.index, unit='ms')
      df_temp = df_temp.set_index('date').resample('s').mean()
      df = pd.concat([df, df_temp])
    clear_output(wait=True)
    print('MFI calc ', i+1,'/ ', days_num + 1,end= ' ')
  print('ok!')
  vars = ['P_DENS','P_VELS','P_TEMP']
  df2 = pd.DataFrame()
  for i in range(days_num + 1):
    if i != days_num:
      status, data = cdas.get_data('WI_PM_3DP', vars, date_start + pd.DateOffset(days=i), date_start + pd.DateOffset(days=i+1))
    else:
      status, data = cdas.get_data('WI_PM_3DP', vars, date_start + pd.DateOffset(days=i), date_end)
    if data is not None:
      data['vx'] = data.P_VELS[:,0]
      data['vy'] = data.P_VELS[:,1]
      data['vz'] = data.P_VELS[:,2]
      data = data.drop('P_VELS')
      data = data.drop('metavar0')
      df_temp = data.to_dataframe()
      df_temp.columns = ['density', 'temp', 'vx', 'vy', 'vz']
      df_temp['date'] = pd.to_datetime(df_temp.index, unit='ms')
      df_temp = df_temp.set_index('date').resample('s').first()
      df2 = pd.concat([df2, df_temp])
    clear_output(wait=True)
    print('plasma params calc ', i+1,'/ ', days_num + 1,end= ' ')
  print('ok!')
  df2 = df2.fillna(method='pad', limit=2)
  df = df.resample('s').first()
  df2 = df2.resample('s').first()
  df = df.iloc[1:-1]
  df2['B_nT'] = df['BF1']
  df = df2
  df['vp'] = (df['vx'] ** 2 + df['vy'] ** 2 + df['vz'] ** 2) ** 0.5
  df['temp'] = df['temp'] * 11604.525
  df['Beta'] = ((4.16 * 0.00001 * df['temp']) + 5.34) * df['density'] / np.square(df['B_nT'])
  df['VA'] = 20 * df['B_nT'] / np.sqrt((df['density']))
  df['B_der'] = np.insert(np.diff(df['B_nT']), 0, np.nan)
  df['Beta_der'] = np.insert(np.diff(df['Beta']), 0, np.nan)
  df['VA/V_der'] = np.insert(np.diff(df['VA'] / df['vp']), 0, np.nan)
  df['current_sheet'] = 0
  df.loc[(df['B_der'] <= -0.14) & ((df['VA/V_der'] <= -0.003) | (df['Beta_der'] >= 0.11)), 'current_sheet'] = 1
  df.to_csv(path)
  return df

# Application

In [ ]:
df = current_sheet_detection(date_start, date_end, path)

plasma params calc  2 /  2 ok!


/usr/local/lib/python3.10/dist-packages/numpy/lib/function_base.py:1452: RuntimeWarning: invalid value encountered in subtract
  a = op(a[slice1], a[slice2])


In [ ]:
df

,density,temp,vx,vy,vz,B_nT,vp,Beta,VA,B_der,Beta_der,VA/V_der,current_sheet,delta_B
date,,,,,,,,,,,,,,
2000-01-01 00:00:01,0.012630,1.739940e+06,-504.705444,230.910034,-6.212505,43.418606,555.054626,0.000521,7726.759277,NaN,NaN,NaN,0,0.000000
2000-01-01 00:00:02,0.012630,1.739940e+06,-504.705444,230.910034,-6.212505,43.598705,555.054626,0.000516,7758.810059,0.180099,-0.000004,0.057744,0,0.004131
2000-01-01 00:00:03,0.012630,1.739940e+06,-504.705444,230.910034,-6.212505,43.734303,555.054626,0.000513,7782.940430,0.135597,-0.000003,0.043474,0,0.003100
2000-01-01 00:00:04,0.014861,2.414007e+05,-258.112000,80.786705,-23.383692,43.934845,271.468414,0.000118,7208.117188,0.200542,-0.000395,12.530388,0,0.004565
2000-01-01 00:00:05,0.014861,2.414007e+05,-258.112000,80.786705,-23.383692,44.183842,271.468414,0.000117,7248.968750,0.248997,-0.000001,0.150482,0,0.005635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2000-01-02 01:04:53,NaN,NaN,-1.929897,0.383880,-0.000000,3.843572,1.967706,NaN,NaN,0.008499,NaN,NaN,0,0.002211
2000-01-02 01:04:54,0.009660,1.001475e+06,-421.384308,5.731064,13.924919,3.855568,421.653290,0.030543,784.565613,0.011996,NaN,NaN,0,0.003111
2000-01-02 01:04:55,0.009660,1.001475e+06,-421.384308,5.731064,13.924919,3.836110,421.653290,0.030854,780.606140,-0.019458,0.000311,-0.009390,0,-0.005072
